[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aranaur/aranaur.rbind.io/blob/main/lectures/kse/MATH840/26autumn/labs/_demo06.ipynb)

# MATH840 — Week 6 practice: ETS, specified by hand and checked by the machine

**This session is not marked.** We build one ETS forecast together, on a series everyone has, and
argue about every decision on the way. Then you do the same on your own series at home — that is
Lab 5, due 23:59 today.

**What is new today: accuracy counts.** Your forecast is scored against your own benchmark on data
nobody has seen: beat it and you get 3 points, come within 10% and you get 2, otherwise 1 with an
explanation.

| Block | What we do | Minutes |
|---|---|---|
| 1 | Split first, find the benchmark | 10 |
| 2 | Specify the model **by hand**, from the plots | 15 |
| 3 | Let `AutoETS` choose, shortlist on AICc, decide on a rolling origin | 20 |
| 4 | Residuals and intervals | 10 |
| 5 | A second series, where the hand and the machine disagree | 10 |
| 6 | Package the submission | 10 |
| 7 | *If there is time:* Walmart, a 52-week season on 2.75 years of data | 15 |

Type along. Every block ends with a question — the answers are the point of the session.

## 0. Setup

In [ ]:
!pip install -q statsforecast==2.1.1   # the version the pool was screened with

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsforecast.models import AutoETS
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.diagnostic import acorr_ljungbox

plt.rcParams.update({"figure.figsize": (11, 4), "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.width", 120, "display.precision", 3)

Our shared series: **Australian gas production**, quarterly, 1956–2010. It comes from the same file as
the Week 4 beer, and no challenge series comes from that file, so nothing we do here hands anyone an
answer.

In [ ]:
BASE = ("https://raw.githubusercontent.com/Aranaur/aranaur.rbind.io/"
        "main/lectures/kse/MATH840/26autumn/data")

production = pd.read_csv(f"{BASE}/aus_production_long.csv", parse_dates=["ds"])


def load(name):
    d = (production.query("unique_id == @name")[["ds", "y"]]
         .dropna().sort_values("ds").reset_index(drop=True))
    return d["y"].to_numpy(float), d["ds"]


M, H = 4, 8                      # quarterly data; the challenge horizon for quarterly series
y, dates = load("Gas")
print(f"{len(y)} quarters, {dates.min().date()} to {dates.max().date()}")

## 1. Split first, find the benchmark

The same helpers as Lab 4, and as your template today. The benchmark is the best of the four simple
methods by RMSE on the last `H` observations — **exactly** the rule the scorer uses, so the number you
see here is the one your forecast will be compared against.

In [ ]:
def benchmark_forecasts(history, h, m):
    T = len(history)
    return {
        "mean":   np.repeat(history.mean(), h),
        "naive":  np.repeat(history[-1], h),
        "snaive": np.array([history[-m + (i % m)] for i in range(h)]),
        "drift":  history[-1] + np.arange(1, h + 1) * (history[-1] - history[0]) / (T - 1),
    }


SCALE = float(np.mean(np.abs(y[M:] - y[:-M])))          # MASE denominator, fixed for this series


def mase(actual, forecast):
    return float(np.mean(np.abs(np.asarray(actual) - np.asarray(forecast))) / SCALE)


def rmse(actual, forecast):
    return float(np.sqrt(np.mean((np.asarray(actual) - np.asarray(forecast)) ** 2)))


train, valid = y[:-H], y[-H:]
train_dates, valid_dates = dates.iloc[:-H], dates.iloc[-H:]

simple = benchmark_forecasts(train, H, M)
scores = pd.DataFrame({name: {"rmse": rmse(valid, fc), "mase": mase(valid, fc)}
                       for name, fc in simple.items()}).T
BENCHMARK = scores["rmse"].idxmin()
print(scores.round(3).to_string())
print(f"\nbenchmark: {BENCHMARK}  (MASE {scores.loc[BENCHMARK, 'mase']:.3f})")

In [ ]:
fig, ax = plt.subplots()
ax.plot(dates.iloc[-40:], y[-40:], color="#314f4f", linewidth=1, label="actual")
for name, fc in simple.items():
    is_bench = name == BENCHMARK
    ax.plot(valid_dates, fc, "-" if is_bench else "--", linewidth=2.2 if is_bench else 1,
            label=f"{name}{' (benchmark)' if is_bench else ''}")
ax.legend(ncol=5)
ax.set_title("The four simple methods over the validation window")
plt.show()

::: Discussion
**The benchmark's MASE on the validation window is above 1. What does that mean — and does it make
the benchmark easy to beat?**
:::

## 2. Specify the model by hand

ETS has three letters — **E**rror, **T**rend, **S**eason — and each one is a question you can answer
from a plot *before* any model is fitted. That is what the justification block asks for first, and
`AutoETS` is the check, not the starting point.

| Letter | Question | Where to look |
|---|---|---|
| T | Does the level move steadily? Does the slope flatten? | Time plot |
| S | Does the seasonal swing grow with the level (M) or stay the same size (A)? | Swing against level |
| E | Do the surprises grow with the level too? | Same plot, the spread around the pattern |

In [ ]:
fig, ax = plt.subplots()
ax.plot(train_dates, train, color="#314f4f", linewidth=1, label="train")
ax.plot(valid_dates, valid, color="#e64173", linewidth=2, label="validation")
ax.legend()
ax.set_title("Australian gas production, petajoules")
plt.show()

The plot that answers the **S** question: one point per year, the seasonal swing (max minus min of the
four quarters) against the level (their mean). A flat cloud says additive; a cloud that climbs with the
level says multiplicative.

In [ ]:
def swing_against_level(values, ds, m):
    years = pd.DataFrame({"year": ds.dt.year.to_numpy(), "y": values})
    full = years.groupby("year").filter(lambda g: len(g) == m)       # complete years only
    per_year = full.groupby("year")["y"].agg(level="mean", swing=lambda v: v.max() - v.min())
    return per_year


per_year = swing_against_level(train, train_dates, M)
ratio = per_year["swing"] / per_year["level"]

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].scatter(per_year["level"], per_year["swing"], color="#20B2AA")
axes[0].set_xlabel("level (mean of the year)")
axes[0].set_ylabel("seasonal swing (max - min)")
axes[0].set_title("Does the swing grow with the level?")
axes[1].plot(per_year.index, ratio, color="#6A5ACD")
axes[1].set_title("Swing as a share of the level")
plt.show()

print(f"correlation of swing with level: {per_year['swing'].corr(per_year['level']):.2f}")
print(f"swing / level, first decade {ratio.iloc[:10].mean():.2f}, last decade {ratio.iloc[-10:].mean():.2f}")

Write the specification down **now**, before the machine speaks. It goes into 3.1 of your justification
block exactly like this: a letter, and the plot that earned it.

In [ ]:
MY_SPEC, MY_DAMPED = "MAM", False   # <-- error, trend, season; and whether the trend is damped

::: Discussion
**The swing tracks the level almost perfectly, and as a share of the level it barely moves. Which letter does that
decide, and what would an additive season do to the forecast here?**

And the harder one: what in the time plot would make you write `Ad` instead of `A`?
:::

## 3. Let `AutoETS` choose — then decide on a rolling origin

Three steps from the Week 4 lecture:

1. **`AutoETS`** searches every admissible model by AICc. Does it agree with you?
2. **Shortlist on AICc**: a handful of specifications, including yours and AutoETS's.
3. **Decide on a rolling origin**: refit on shorter histories and score the next `H` quarters,
   several times. One validation window is one draw; three are less of an accident.

In [ ]:
def fit_ets(history, spec=None, damped=None):
    # spec=None lets AutoETS search. "AAdA" is read as "AAA" with damped=True; a trend of "N" cannot be damped.
    if spec is None:
        return AutoETS(season_length=M).fit(y=history)
    spec = str(spec).strip().upper()
    if spec.endswith("DA") or spec.endswith("DM") or spec.endswith("DN"):
        spec, damped = spec[:2] + spec[3:], True
    if len(spec) != 3 or spec[0] not in "AM" or spec[1] not in "NA" or spec[2] not in "ANM":
        raise ValueError(f"spec must be three letters - error (A/M), trend (N/A), season (N/A/M), "
                         f"e.g. 'MAM'; damping goes in `damped` or as 'AAdA'. Got {spec!r}")
    return AutoETS(season_length=M, model=spec,
                   damped=bool(damped) if spec[1] != "N" else None).fit(y=history)


auto = fit_ets(train)
mine = fit_ets(train, MY_SPEC, MY_DAMPED)
print("AutoETS chose:", auto.model_["method"], "  AICc", round(auto.model_["aicc"], 1))
print("you specified:", mine.model_["method"], "  AICc", round(mine.model_["aicc"], 1))

In [ ]:
states = auto.model_["states"]
fig, axes = plt.subplots(4, 1, figsize=(11, 7), sharex=True)
axes[0].plot(train_dates, train, color="#314f4f", linewidth=1)
axes[0].set_title(f"{auto.model_['method']}: what each letter estimated on the training data")
axes[0].set_ylabel("y")
axes[1].plot(train_dates, states[1:, 0], color="#181485")
axes[1].set_ylabel("level")
axes[2].plot(train_dates, states[1:, 1], color="#20B2AA")
axes[2].set_ylabel("slope")
axes[3].plot(train_dates, states[1:, 2], color="#FFA500")
axes[3].set_ylabel("season\n(multiplier)")
plt.tight_layout()
plt.show()

In [ ]:
def rolling_mase(spec, damped, history, h, windows=3):
    # The last window is the validation window; each earlier one ends h steps before the next.
    errors = []
    for k in range(windows):
        end = len(history) - k * h
        fit = fit_ets(history[:end - h], spec, damped)
        errors.append(mase(history[end - h:end], fit.predict(h=h)["mean"]))
    return float(np.mean(errors))


SHORTLIST = [(None, None), ("MAM", False), ("MAM", True), ("MNM", None),
             ("AAA", False), ("AAA", True), ("ANA", None)]

rows = []
for spec, damped in SHORTLIST:
    fit = fit_ets(train, spec, damped)
    rows.append({"spec": "AutoETS" if spec is None else spec + (" damped" if damped else ""),
                 "model": fit.model_["method"],
                 "AICc": fit.model_["aicc"],
                 "valid MASE": mase(valid, fit.predict(h=H)["mean"]),
                 "rolling MASE (3 windows)": rolling_mase(spec, damped, y, H)})
table = pd.DataFrame(rows).sort_values("AICc")
print(table.round(3).to_string(index=False))

# The working rule from the Week 4 lecture: shortlist the three best models by AICc, decide by the rolling origin.
shortlist = table.drop_duplicates("model").nsmallest(3, "AICc")
choice = shortlist.loc[shortlist["rolling MASE (3 windows)"].idxmin()]
print(f"\nshortlist by AICc: {', '.join(shortlist['model'])}  ->  rolling origin picks {choice['model']}")

In [ ]:
def rolling_errors(spec, damped, history, h, windows=3):
    # The same windows as rolling_mase, one number per window, oldest first.
    out = []
    for k in reversed(range(windows)):
        end = len(history) - k * h
        fit = fit_ets(history[:end - h], spec, damped)
        out.append(mase(history[end - h:end], fit.predict(h=h)["mean"]))
    return out


show = [("MAM", False), ("MAM", True), ("AAA", False), ("MNM", None)]
label = lambda spec, damped: fit_ets(train, spec, damped).model_["method"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
axes[0].plot(dates.iloc[-24:], y[-24:], color="#314f4f", linewidth=1.2, label="actual")
for spec, damped in show:
    axes[0].plot(valid_dates, fit_ets(train, spec, damped).predict(h=H)["mean"], linewidth=1.6,
                 label=label(spec, damped))
axes[0].plot(valid_dates, simple[BENCHMARK], ":", color="#e64173", linewidth=2, label=BENCHMARK)
axes[0].legend(fontsize=8)
axes[0].set_title("One window: the validation quarters")

width = 0.2
for j, (spec, damped) in enumerate(show):
    axes[1].bar(np.arange(3) + (j - 1.5) * width, rolling_errors(spec, damped, y, H), width,
                label=label(spec, damped))
axes[1].set_xticks(range(3), ["window 1\n(oldest)", "window 2", "window 3\n= validation"])
axes[1].axhline(1, color="grey", linewidth=0.8)
axes[1].set_ylabel("MASE")
axes[1].legend(fontsize=8)
axes[1].set_title("Three windows: the ranking is not stable")
plt.show()

::: Discussion
**On the single validation window the damped model looks best. Over three windows it is the worst of
the shortlist. Which number would you have trusted if you had computed only one — and what does that
say about choosing on one window?**

And: `AAA` has the best rolling MASE of all, yet AICc puts it 160 units behind. Is a difference of 0.04
in rolling MASE evidence, or noise?

One fact you need for that answer. The pool was screened so that **`AutoETS` fitted on your whole
history** beats your benchmark on the hidden horizon — that is the guarantee from Week 1, and it is a
guarantee about `AutoETS`, not about ETS in general. Choosing something else is legitimate and is
exactly what the rubric pays you to argue; it also means you are leaving the guaranteed route on your
own evidence. Measured on the whole pool: keeping the three best by AICc and choosing among them by a
rolling origin beats the benchmark on 94% of series; choosing by the rolling origin among everything,
85%.
:::

## 4. Residuals and intervals

ETS residuals come in two kinds (Week 4 lecture): the **innovation** residuals are the $\varepsilon_t$
the model claims are white noise, and those are the ones to test. For a multiplicative model they are
*relative* errors, not petajoules.

In [ ]:
FINAL_SPEC, FINAL_DAMPED = "MAM", False       # <-- what the rule in block 3 picked
final_train = fit_ets(train, FINAL_SPEC, FINAL_DAMPED)
resid = final_train.model_["residuals"]

fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
axes[0].plot(train_dates, resid, color="#FFA500", linewidth=0.9)
axes[0].set_title(f"Innovation residuals, {final_train.model_['method']}")
plot_acf(resid, lags=16, zero=False, ax=axes[1], title="ACF of residuals")
axes[2].hist(resid, bins=25, color="#20B2AA")
axes[2].set_title("Distribution")
plt.tight_layout()
plt.show()

print(acorr_ljungbox(resid, lags=[2 * M]).round(4).to_string())

In [ ]:
fc_valid = final_train.predict(h=H, level=[80])
inside = (valid >= fc_valid["lo-80"]) & (valid <= fc_valid["hi-80"])
print(f"80% interval covered {inside.sum()} of {H} validation quarters")

fig, ax = plt.subplots()
ax.plot(dates.iloc[-40:], y[-40:], color="#314f4f", linewidth=1, label="actual")
ax.plot(valid_dates, fc_valid["mean"], color="#20B2AA", linewidth=2, label=final_train.model_["method"])
ax.fill_between(valid_dates, fc_valid["lo-80"], fc_valid["hi-80"], color="#20B2AA", alpha=0.2,
                label="80% interval")
ax.plot(valid_dates, simple[BENCHMARK], "--", color="#e64173", linewidth=2, label=f"benchmark: {BENCHMARK}")
ax.legend()
ax.set_title("The chosen model over the validation window")
plt.show()

::: Discussion
**Ljung-Box rejects. Does that mean the forecast is wrong, the interval is wrong, or neither?**

And: the interval covered 8 out of 8. Is that good news? What would 8 out of 8 look like for an
interval that is far too wide?
:::

## 5. A second series: when the hand and the machine disagree

The Week 4 beer. On the time plot its season looks stable and its trend turned down in the 1970s, so
a hand specification read off that plot alone says additive season, damped trend: `AAA`, damped. Here is
what the machine says — and the Week 4 decomposition forecast to compare against: **STL + naive, MASE
0.734** on this same validation window. The plot after the table is the one the hand skipped.

In [ ]:
yb, dates_b = load("Beer")
scale_b = float(np.mean(np.abs(yb[M:] - yb[:-M])))
train_b, valid_b = yb[:-H], yb[-H:]
mase_b = lambda fc: float(np.mean(np.abs(valid_b - np.asarray(fc))) / scale_b)

rows = []
for spec, damped in [(None, None), ("AAA", True), ("MAM", False), ("MNM", None), ("ANA", None)]:
    fit = fit_ets(train_b, spec, damped)
    rows.append({"spec": "AutoETS" if spec is None else spec + (" damped" if damped else ""),
                 "model": fit.model_["method"], "AICc": fit.model_["aicc"],
                 "valid MASE": mase_b(fit.predict(h=H)["mean"])})
rows.append({"spec": "Week 4: STL + naive", "model": "-", "AICc": np.nan, "valid MASE": 0.734})
rows.append({"spec": "snaive (benchmark)", "model": "-", "AICc": np.nan,
             "valid MASE": mase_b(benchmark_forecasts(train_b, H, M)["snaive"])})
print(pd.DataFrame(rows).sort_values("valid MASE").round(3).to_string(index=False))

In [ ]:
per_year_b = swing_against_level(train_b, dates_b.iloc[:-H], M)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
points = axes[0].scatter(per_year_b["level"], per_year_b["swing"], c=per_year_b.index, cmap="viridis")
fig.colorbar(points, ax=axes[0], label="year")
axes[0].set_xlabel("level (mean of the year)")
axes[0].set_ylabel("seasonal swing (max - min)")
axes[0].set_title(f"Beer: swing against level, corr {per_year_b['swing'].corr(per_year_b['level']):.2f}")

axes[1].plot(dates_b.iloc[-24:], yb[-24:], color="#314f4f", linewidth=1.2, label="actual")
for spec, damped, colour in [(None, None, "#20B2AA"), ("AAA", True, "#6A5ACD")]:
    fit = fit_ets(train_b, spec, damped)
    axes[1].plot(dates_b.iloc[-H:], fit.predict(h=H)["mean"], color=colour, linewidth=1.8,
                 label=f"{fit.model_['method']}{' (AutoETS)' if spec is None else ' (by hand)'}")
axes[1].plot(dates_b.iloc[-H:], benchmark_forecasts(train_b, H, M)["snaive"], ":", color="#e64173",
             linewidth=2, label="snaive")
axes[1].legend()
axes[1].set_title("Two specifications, almost the same forecast")
plt.show()

::: Discussion
**AutoETS chose a multiplicative model where the hand said additive — and the swing-against-level
plot, which the hand never looked at, leans the machine's way (corr 0.58). Yet on the validation window
the two forecasts are almost the same line. Which statement goes into 3.2 — "AutoETS was wrong", "I was
wrong", or something else?**

The lesson for 3.1: the time plot answers **T**; it does not answer **S**. Draw the plot that does.

And: ETS barely improves on the Week 4 decomposition forecast here. What does that say about how much
of beer is level-and-season, and how much is something neither method models?
:::

## 6. Package the submission

Now the validation window goes back into the data. Refit the chosen specification on the **whole**
history and let the model produce its own 80% interval — no more hand-made intervals: from today the
interval comes from the model.

In [ ]:
final = fit_ets(y, FINAL_SPEC, FINAL_DAMPED)
fc = final.predict(h=H, level=[80])
FUTURE = pd.date_range(dates.iloc[-1], periods=H + 1, freq=pd.infer_freq(dates))[1:]

submission = pd.DataFrame({
    "student_id": "DEMO",
    "code": "DEMO-GAS",      # yours will be your series code
    "ds": FUTURE.strftime("%Y-%m-%d"),
    "yhat": fc["mean"],
    "yhat_lo_80": fc["lo-80"],
    "yhat_hi_80": fc["hi-80"],
})
submission.to_csv("DEMO_lab06_forecast.csv", index=False)

check = pd.read_csv("DEMO_lab06_forecast.csv", parse_dates=["ds"])
assert len(check) == H
assert list(check.columns) == ["student_id", "code", "ds", "yhat", "yhat_lo_80", "yhat_hi_80"]
assert check[["yhat", "yhat_lo_80", "yhat_hi_80"]].notna().all().all()
print(final.model_["method"], "fitted on all", len(y), "quarters")
print(submission.to_string(index=False))
print("\nformat OK")

In [ ]:
fig, ax = plt.subplots()
ax.plot(dates.iloc[-40:], y[-40:], color="#314f4f", linewidth=1, label="history")
ax.plot(FUTURE, fc["mean"], color="#20B2AA", linewidth=2, label="forecast")
ax.fill_between(FUTURE, fc["lo-80"], fc["hi-80"], color="#20B2AA", alpha=0.2, label="80% interval")
ax.legend()
ax.set_title("What we would submit")
plt.show()

::: Discussion
**The final model is refitted on all the data. Will it still be the same ETS — the same letters, the
same parameters? Should you check, and what do you do if the letters change?**
:::

## 7. If there is time: Walmart weekly sales, where the season is 52 long

The Walmart chain from the Week 4 practice: 45 stores added up, weekly, February 2010 to October 2012.
Two things make it the opposite of gas: the season is **52** weeks long, and the history is only
**2.75** seasons. Same split as Week 4 — the last 13 weeks held out — and the same scale for MASE.

In [ ]:
walmart = pd.read_csv(f"{BASE}/walmart.csv", parse_dates=["ds"])
chain = walmart.groupby("ds", as_index=False)["y"].sum()
yw = chain["y"].to_numpy(float) / 1e6                      # millions of dollars per week
MW, HW = 52, 13
train_w, valid_w = yw[:-HW], yw[-HW:]
scale_w = float(np.mean(np.abs(yw[MW:] - yw[:-MW])))
mase_w = lambda fc: float(np.mean(np.abs(valid_w - np.asarray(fc))) / scale_w)

simple_w = benchmark_forecasts(train_w, HW, MW)
print(f"{len(yw)} weeks = {len(yw) / MW:.2f} seasons; training on {len(train_w)} weeks")
print({name: round(mase_w(fc), 3) for name, fc in simple_w.items()})

In [ ]:
weeks = chain.assign(y=yw, year=chain["ds"].dt.year, week=chain["ds"].dt.isocalendar().week.astype(int))
fig, axes = plt.subplots(1, 2, figsize=(13, 4), width_ratios=[1.6, 1])
axes[0].plot(chain["ds"].iloc[:-HW], train_w, color="#314f4f", linewidth=1, label="train")
axes[0].plot(chain["ds"].iloc[-HW:], valid_w, color="#e64173", linewidth=2, label="validation")
axes[0].legend()
axes[0].set_ylabel("$m per week")
axes[0].set_title("Walmart chain, weekly sales")
for year, colour in zip((2010, 2011, 2012), ["#20B2AA", "#FFA500", "#181485"]):
    d = weeks[weeks["year"] == year]
    axes[1].plot(d["week"], d["y"], color=colour, linewidth=1.2, label=str(year))
axes[1].set_xlabel("week of the year")
axes[1].legend()
axes[1].set_title("Every season we have: two and three quarters")
plt.show()

**Move 1: just ask `AutoETS` for a weekly season.** It does not refuse.

In [ ]:
ets52 = AutoETS(season_length=MW).fit(y=train_w)
ann = AutoETS(season_length=MW, model="ANN").fit(y=train_w)
print(f"AutoETS, m = 52: {ets52.model_['method']}   parameters {ets52.model_['n_params']} "
      f"on {len(train_w)} weeks   AICc {ets52.model_['aicc']:.1f}  (no season: {ann.model_['aicc']:.1f})")
print(f"validation MASE {mase_w(ets52.predict(h=HW)['mean']):.3f}   vs snaive {mase_w(simple_w['snaive']):.3f}")

In [ ]:
fig, ax = plt.subplots()
ax.plot(chain["ds"].iloc[-60:], yw[-60:], color="#314f4f", linewidth=1.2, label="actual")
ax.plot(chain["ds"].iloc[-HW:], ets52.predict(h=HW)["mean"], color="#6A5ACD", linewidth=2,
        label=f"{ets52.model_['method']}, m = 52")
ax.plot(chain["ds"].iloc[-HW:], simple_w["snaive"], ":", color="#e64173", linewidth=2, label="snaive")
ax.legend()
ax.set_ylabel("$m per week")
ax.set_title("The 52-week ETS: last year's shape, smoothed and shifted")
plt.show()

::: Discussion
**AICc prefers the seasonal model by a wide margin, and it still loses to seasonal naive. How can both be
true?**

Count: 51 seasonal states, each estimated from two or three values of the same week. What is the
difference between that and copying last year?
:::

**Move 2: take the season out first, smooth what is left.** STL estimates the 52-week shape once;
ETS only has to follow the seasonally adjusted level, which is the job it does well. The season is
carried forward as in Week 4.

In [ ]:
from statsmodels.tsa.seasonal import STL


def stl_ets(history, h, m, spec=None):
    # STL removes the season; a non-seasonal ETS forecasts the adjusted series; the season goes back.
    stl = STL(history, period=m, seasonal=13, robust=True).fit()
    adjusted = history - stl.seasonal
    season = np.array([stl.seasonal[-m + (i % m)] for i in range(h)])
    ets = (AutoETS(season_length=1) if spec is None else AutoETS(season_length=1, model=spec)).fit(y=adjusted)
    fc = ets.predict(h=h, level=[80])
    return fc["mean"] + season, fc["lo-80"] + season, fc["hi-80"] + season, ets


def stl_naive(history, h, m):
    stl = STL(history, period=m, seasonal=13, robust=True).fit()
    return (history - stl.seasonal)[-1] + np.array([stl.seasonal[-m + (i % m)] for i in range(h)])


rows = []
for label, fc in [("snaive (benchmark)", simple_w["snaive"]),
                  ("AutoETS, m = 52", ets52.predict(h=HW)["mean"]),
                  ("Week 4: STL + naive", stl_naive(train_w, HW, MW))]:
    rows.append({"forecast": label, "valid MASE": mase_w(fc), "80% covered": "-"})
for spec in (None, "ANN"):
    mean, lo, hi, ets = stl_ets(train_w, HW, MW, spec)
    rows.append({"forecast": f"STL + {ets.model_['method']}", "valid MASE": mase_w(mean),
                 "80% covered": f"{int(((valid_w >= lo) & (valid_w <= hi)).sum())} of {HW}"})
print(pd.DataFrame(rows).round(3).to_string(index=False))

In [ ]:
mean, lo, hi, ets = stl_ets(train_w, HW, MW)
fig, ax = plt.subplots()
ax.plot(chain["ds"].iloc[-60:], yw[-60:], color="#314f4f", linewidth=1.2, label="actual")
ax.plot(chain["ds"].iloc[-HW:], mean, color="#20B2AA", linewidth=2, label=f"STL + {ets.model_['method']}")
ax.fill_between(chain["ds"].iloc[-HW:], lo, hi, color="#20B2AA", alpha=0.2, label="80% interval")
ax.plot(chain["ds"].iloc[-HW:], simple_w["snaive"], ":", color="#e64173", linewidth=2, label="snaive")
ax.legend()
ax.set_ylabel("$m per week")
ax.set_title("Decompose first: closer to the actuals, and an interval that is too narrow")
plt.show()

One window is one draw, so the same three on a rolling origin — three windows of 13 weeks. The earliest
one trains on two seasons exactly, which is as little as STL can work with.

In [ ]:
def rolling_w(forecaster, windows=3):
    errors = []
    for k in range(windows):
        end = len(yw) - k * HW
        fc = forecaster(yw[:end - HW])
        errors.append(float(np.mean(np.abs(yw[end - HW:end] - fc)) / scale_w))
    return np.round(errors, 3), round(float(np.mean(errors)), 3)


for label, forecaster in [("snaive", lambda hist: benchmark_forecasts(hist, HW, MW)["snaive"]),
                          ("STL + naive", lambda hist: stl_naive(hist, HW, MW)),
                          ("STL + AutoETS", lambda hist: stl_ets(hist, HW, MW)[0])]:
    per_window, mean = rolling_w(forecaster)
    print(f"{label:14s} windows {per_window}  mean {mean}")

In [ ]:
results = {label: rolling_w(forecaster)[0] for label, forecaster in
           [("snaive", lambda hist: benchmark_forecasts(hist, HW, MW)["snaive"]),
            ("STL + naive", lambda hist: stl_naive(hist, HW, MW)),
            ("STL + AutoETS", lambda hist: stl_ets(hist, HW, MW)[0])]}
fig, ax = plt.subplots(figsize=(9, 3.8))
width = 0.27
for j, (label, errs) in enumerate(results.items()):
    ax.bar(np.arange(3) + (j - 1) * width, errs[::-1], width, label=label)
ax.set_xticks(range(3), ["window 1\n(oldest)", "window 2", "window 3\n= validation"])
ax.set_ylabel("MASE")
ax.legend()
ax.set_title("Walmart, three windows of 13 weeks")
plt.show()

::: Discussion
**Decomposing first beats seasonal naive on all three windows. Does ETS deserve the credit, or STL?**
Compare STL + AutoETS with STL + naive before you answer.

And the intervals: STL + ETS covered well under 80% of the validation weeks. What does the interval
leave out? (Hint: who estimated the seasonal component, and is its uncertainty in the interval?)

Your own series is monthly or quarterly, with fifteen years or more — none of this bites there. Long and
multiple seasonal periods are Week 8.
:::

### What to do at home

Lab 5 is this notebook, on your series, with an argument attached:

1. the plots, including the one that decides the season — **Section 1** (1 point);
2. your specification by hand, then `AutoETS`, the shortlist and the rolling origin, the residuals and
   the interval — **Section 3, the justification block** (3 points);
3. a submission file written by the notebook — scored on the hidden horizon against your benchmark
   (**3 points for accuracy**);
4. a notebook that runs top to bottom — **Section 4** (1 point).

The template has the same helpers as this notebook:
[Lab 5 template](https://colab.research.google.com/github/Aranaur/aranaur.rbind.io/blob/main/lectures/kse/MATH840/26autumn/labs/_lab06.ipynb).